# Imports

In [1]:
import xarray as xr
from dask_jobqueue import PBSCluster
from dask.distributed import Client
import numpy as np
from scipy.stats import t
import dask

In [2]:
dask.config.set({'array.chunk-size': '8 GiB'})
cluster = PBSCluster(
    cores=1, # The number of cores you want
    memory='32GB', # Amount of memory (resource_spec is the one)
    processes=1, # How many processes
    queue='casper', # The type of queue to utilize (/glade/u/apps/dav/opt/usr/bin/execcasper)
    local_directory='$TMPDIR', # Use your local directory
    account='P93300313', # Input your project ID here
    walltime='01:00:00', # Amount of wall time
)
cluster.scale(jobs=4)
client = Client(cluster)
client

Connection method: Cluster object,Cluster type: dask_jobqueue.PBSCluster
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/acruz/Analysis/proxy/8787/status,
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/acruz/Analysis/proxy/8787/status,Workers: 0
Total threads: 0,Total memory: 0 B
Comm: tcp://128.117.208.175:44429,Workers: 0
Dashboard: https://jupyterhub.hpc.ucar.edu/stable/user/acruz/Analysis/proxy/8787/status,Total threads: 0
Started: Just now,Total memory: 0 B


# function

In [3]:
def xr_regression(x, y, lag_x=0, lag_y=0, dim="time", alternative="two-sided"):
    """
    From https://stackoverflow.com/questions/52108417/how-to-apply-linear-regression-to-every-pixel-in-a-large-multi-dimensional-array
    requires scipy.stats as t
    Takes two xr.Datarrays of any dimensions (input data could be a 1D
    time series, or for example, have three dimensions e.g. time, lat,
    lon), and returns covariance, correlation, coefficient of
    determination, regression slope, intercept, p-value and standard
    error, and number of valid observations (n) between the two datasets
    along their aligned first dimension.

    Datasets can be provided in any order, but note that the regression
    slope and intercept will be calculated for y with respect to x.

    Inspired by:
    https://hrishichandanpurkar.blogspot.com/2017/09/vectorized-functions-for-correlation.html

    Parameters
    ----------
    x, y : xarray DataArray
        Two xarray DataArrays with any number of dimensions, both
        sharing the same first dimension
    lag_x, lag_y : int, optional
        Optional integers giving lag values to assign to either of the
        data, with lagx shifting x, and lagy shifting y with the
        specified lag amount.
    dim : str, optional
        An optional string giving the name of the dimension on which to
        align (and optionally lag) datasets. The default is 'time'.
    alternative : string, optional
        Defines the alternative hypothesis. Default is 'two-sided'.
        The following options are available:

        * 'two-sided': slope of the regression line is nonzero
        * 'less': slope of the regression line is less than zero
        * 'greater':  slope of the regression line is greater than zero

    Returns
    -------
    regression_ds : xarray.Dataset
        A dataset comparing the two input datasets along their aligned
        dimension, containing variables including covariance, correlation,
        coefficient of determination, regression slope, intercept,
        p-value and standard error, and number of valid observations (n).

    """

    # Shift x and y data if lags are specified
    if lag_x != 0:
        # If x lags y by 1, x must be shifted 1 step backwards. But as
        # the 'zero-th' value is nonexistant, xarray assigns it as
        # invalid (nan). Hence it needs to be dropped
        x = x.shift(**{dim: -lag_x}).dropna(dim=dim)

        # Next re-align the two datasets so that y adjusts to the
        # changed coordinates of x
        x, y = xr.align(x, y)

    if lag_y != 0:
        y = y.shift(**{dim: -lag_y}).dropna(dim=dim)

    # Ensure that the data are properly aligned to each other.
    x, y = xr.align(x, y)

    # Compute data length, mean and standard deviation along dim
    n = y.notnull().sum(dim=dim)
    xmean = x.mean(dim=dim)
    ymean = y.mean(dim=dim)
    xstd = x.std(dim=dim)
    ystd = y.std(dim=dim)

    # Compute covariance, correlation and coefficient of determination
    cov = ((x - xmean) * (y - ymean)).sum(dim=dim) / (n)
    cor = cov / (xstd * ystd)
    r2 = cor**2

    # Compute regression slope and intercept
    slope = cov / (xstd**2)
    intercept = ymean - xmean * slope

    # Compute t-statistics and standard error
    tstats = cor * np.sqrt(n - 2) / np.sqrt(1 - cor**2)
    stderr = slope / tstats

    # Calculate p-values for different alternative hypotheses.
    if alternative == "two-sided":
        pval = t.sf(np.abs(tstats), n - 2) * 2
    elif alternative == "greater":
        pval = t.sf(tstats, n - 2)
    elif alternative == "less":
        pval = t.cdf(np.abs(tstats), n - 2)

    # Wrap p-values into an xr.DataArray
    pval = xr.DataArray(pval, dims=cor.dims, coords=cor.coords)

    # Combine into single dataset
    regression_ds = xr.merge(
        [
            cov.rename("cov").astype(np.float32),
            cor.rename("cor").astype(np.float32),
            r2.rename("r2").astype(np.float32),
            slope.rename("slope").astype(np.float32),
            intercept.rename("intercept").astype(np.float32),
            pval.rename("pvalue").astype(np.float32),
            stderr.rename("stderr").astype(np.float32),
            n.rename("n").astype(np.int16),
        ]
    )

    return regression_ds

# Data Import

In [4]:
EOF_ds = xr.open_dataset("/glade/work/acruz/CESM/EANI_CANI_CESM.nc")
EOD_ds = EOF_ds.chunk(chunks='auto')
EOF_ds

<xarray.Dataset> Size: 2MB
Dimensions:  (member: 100, time: 1212)
Coordinates:
  * member   (member) int64 800B 0 1 2 3 4 5 6 7 8 ... 92 93 94 95 96 97 98 99
  * time     (time) object 10kB 1914-01-01 00:00:00 ... 2014-12-01 00:00:00
    month    (time) int64 10kB ...
Data variables:
    EANI     (member, time) float64 970kB ...
    CANI     (member, time) float64 970kB ...
Attributes:
    title:        Atlantic Niño Index Timeseries
    description:  Contains EANI, CANI calculated from CESM2.1 LE2
    created:      12/25/2025

In [5]:
ssta_ds = xr.open_zarr('/glade/work/acruz/CESM/anomalies/SSTA')
ssta_ds = ssta_ds.chunk(chunks='auto')
ssta_ds = ssta_ds.sel(time=slice('1914-01-01', '2014-12-01'))
ssta_ds.coords['lon'] = (ssta_ds.coords['lon'] + 180) % 360 - 180
ssta_ds = ssta_ds.sortby(ssta_ds.lon)
ssta_ds = ssta_ds.assign_coords({'member': ssta_ds['member']})
ssta_ds

<xarray.Dataset> Size: 27GB
Dimensions:  (member: 100, time: 1212, lat: 192, lon: 288)
Coordinates:
  * member   (member) int64 800B 0 1 2 3 4 5 6 7 8 ... 92 93 94 95 96 97 98 99
  * time     (time) object 10kB 1914-01-01 00:00:00 ... 2014-12-01 00:00:00
  * lat      (lat) float64 2kB -90.0 -89.06 -88.12 -87.17 ... 88.12 89.06 90.0
  * lon      (lon) float64 2kB -180.0 -178.8 -177.5 -176.2 ... 176.2 177.5 178.8
    month    (time) int64 10kB dask.array<chunksize=(1212,), meta=np.ndarray>
Data variables:
    SST      (member, time, lat, lon) float32 27GB dask.array<chunksize=(100, 337, 192, 288), meta=np.ndarray>

# Data selection

In [6]:
dates = slice('1914-01-01', '2014-12-31')
ssta_ds = ssta_ds.sel(time=dates)
EOF_ds = EOF_ds.sel(time=dates)
EOF_ds

<xarray.Dataset> Size: 2MB
Dimensions:  (member: 100, time: 1212)
Coordinates:
  * member   (member) int64 800B 0 1 2 3 4 5 6 7 8 ... 92 93 94 95 96 97 98 99
  * time     (time) object 10kB 1914-01-01 00:00:00 ... 2014-12-01 00:00:00
    month    (time) int64 10kB ...
Data variables:
    EANI     (member, time) float64 970kB ...
    CANI     (member, time) float64 970kB ...
Attributes:
    title:        Atlantic Niño Index Timeseries
    description:  Contains EANI, CANI calculated from CESM2.1 LE2
    created:      12/25/2025

# Seasonal Peak Mean

In [7]:
JJAds = ssta_ds['SST'].sel(time=ssta_ds.time.dt.month.isin([6, 7, 8])).groupby('time.year').mean()
SONds = ssta_ds['SST'].sel(time=ssta_ds.time.dt.month.isin([9, 10, 11])).groupby('time.year').mean()

EANI = EOF_ds['EANI'].sel(time=EOF_ds['EANI'].time.dt.month.isin([6, 7, 8])).groupby('time.year').mean()
CANI = EOF_ds['CANI'].sel(time=EOF_ds['CANI'].time.dt.month.isin([6, 7, 8])).groupby('time.year').mean()

In [8]:
JJAds

<xarray.DataArray 'SST' (member: 100, year: 101, lat: 192, lon: 288)> Size: 2GB
dask.array<transpose, shape=(100, 101, 192, 288), dtype=float32, chunksize=(100, 1, 192, 288), chunktype=numpy.ndarray>
Coordinates:
  * member   (member) int64 800B 0 1 2 3 4 5 6 7 8 ... 92 93 94 95 96 97 98 99
  * year     (year) int64 808B 1914 1915 1916 1917 1918 ... 2011 2012 2013 2014
  * lat      (lat) float64 2kB -90.0 -89.06 -88.12 -87.17 ... 88.12 89.06 90.0
  * lon      (lon) float64 2kB -180.0 -178.8 -177.5 -176.2 ... 176.2 177.5 178.8
Attributes:
    units:         K
    long_name:     sea surface temperature
    cell_methods:  time: mean

In [9]:
EANI

<xarray.DataArray 'EANI' (member: 100, year: 101)> Size: 81kB
array([[-0.20437016,  0.50830728, -0.50642889, ...,  0.50578362,
        -0.06054163,  1.2895827 ],
       [-0.90714462,  0.30785695, -0.32273305, ..., -0.17559215,
         0.83625258,  1.28752889],
       [-0.20208989,  1.28374879,  0.1835642 , ...,  0.30183476,
        -0.47843454,  1.67845338],
       ...,
       [-1.16473837, -1.11544291, -0.14299141, ...,  0.29730741,
         1.69669882, -0.49405776],
       [-0.0651627 ,  0.32949653,  1.14232025, ...,  0.51885885,
         0.02950817,  0.60261124],
       [ 0.68712636,  0.48914587,  1.16746123, ...,  0.9698736 ,
         0.07525145,  0.82923271]], shape=(100, 101))
Coordinates:
  * member   (member) int64 800B 0 1 2 3 4 5 6 7 8 ... 92 93 94 95 96 97 98 99
  * year     (year) int64 808B 1914 1915 1916 1917 1918 ... 2011 2012 2013 2014

# Regression

In [10]:
JJA_EANI = xr_regression(EANI, JJAds, dim='year')
JJA_CANI = xr_regression(CANI, JJAds, dim='year')
SON_EANI = xr_regression(EANI, SONds, dim='year')
SON_CANI = xr_regression(CANI, SONds, dim='year')

In [11]:
regression = xr.Dataset(
    data_vars={
        "JJA_SSTA_EANI_slope": (['member', 'lat', 'lon'], JJA_EANI['slope'].data),
        "JJA_SSTA_EANI_pval": (['member', 'lat', 'lon'], JJA_EANI['pvalue'].data),
        "JJA_SSTA_CANI_slope": (['member', 'lat', 'lon'], JJA_CANI['slope'].data),
        "JJA_SSTA_CANI_pval": (['member', 'lat', 'lon'], JJA_CANI['pvalue'].data),
        "SON_SSTA_EANI_slope": (['member', 'lat', 'lon'], SON_EANI['slope'].data),
        "SON_SSTA_EANI_pval": (['member', 'lat', 'lon'], SON_EANI['pvalue'].data),
        "SON_SSTA_CANI_slope": (['member', 'lat', 'lon'], SON_CANI['slope'].data),
        "SON_SSTA_CANI_pval": (['member', 'lat', 'lon'], SON_CANI['pvalue'].data)
    },
    coords={
        "member": JJA_EANI['member'].data,
        "lat": JJA_EANI['lat'].data,
        "lon": JJA_EANI['lon'].data,
    },
    attrs={
        "Description": "Linear regression results from CESM SST anomaalies between 1914-2014"
    }
)
regression

<xarray.Dataset> Size: 177MB
Dimensions:              (member: 100, lat: 192, lon: 288)
Coordinates:
  * member               (member) int64 800B 0 1 2 3 4 5 6 ... 94 95 96 97 98 99
  * lat                  (lat) float64 2kB -90.0 -89.06 -88.12 ... 89.06 90.0
  * lon                  (lon) float64 2kB -180.0 -178.8 -177.5 ... 177.5 178.8
Data variables:
    JJA_SSTA_EANI_slope  (member, lat, lon) float32 22MB dask.array<chunksize=(100, 192, 288), meta=np.ndarray>
    JJA_SSTA_EANI_pval   (member, lat, lon) float32 22MB nan nan ... 0.814
    JJA_SSTA_CANI_slope  (member, lat, lon) float32 22MB dask.array<chunksize=(100, 192, 288), meta=np.ndarray>
    JJA_SSTA_CANI_pval   (member, lat, lon) float32 22MB nan nan ... 0.4829
    SON_SSTA_EANI_slope  (member, lat, lon) float32 22MB dask.array<chunksize=(100, 192, 288), meta=np.ndarray>
    SON_SSTA_EANI_pval   (member, lat, lon) float32 22MB nan nan ... 0.1475
    SON_SSTA_CANI_slope  (member, lat, lon) float32 22MB dask.array<chunksize=(100, 192, 288), meta=np.ndarray>
    SON_SSTA_CANI_pval   (member, lat, lon) float32 22MB nan nan ... 0.8544
Attributes:
    Description:  Linear regression results from CESM SST anomaalies between ...

In [12]:
regression.to_zarr('/glade/work/acruz/CESM/SSTAreg', mode='w', consolidated=True)

/glade/u/home/acruz/.conda/envs/analysis_11_20_2025/lib/python3.11/site-packages/zarr/api/asynchronous.py:247: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


In [3]:
client.shutdown()